In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor ,XGBClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score

In [ ]:
train_df = pd.read_csv("/kaggle/input/llm-classification-finetuning/train.csv")
test_df = pd.read_csv("/kaggle/input/llm-classification-finetuning/test.csv")


In [ ]:
train_df.head()

In [ ]:
train_df.isna().sum()

In [ ]:
train_df["text"] = train_df["prompt"] + ' ' + train_df["response_a"] + ' ' + train_df["response_b"]

In [ ]:
vectorizer = TfidfVectorizer(max_features=1200)
X = vectorizer.fit_transform(train_df['text'])


In [ ]:
y  = train_df[["winner_model_a", "winner_model_b", "winner_tie"]]
y = np.argmax(y, axis = 1)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
model = XGBClassifier(
    n_estimators=300,        # number of trees
    learning_rate=0.05,      # smaller step size
    max_depth=5,             # limit depth to avoid overfitting
    min_child_weight=1,      
    subsample=0.8,           # use 80% samples per tree
    colsample_bytree=0.8,    # use 80% features per tree
    gamma=0,                 
    reg_alpha=0,             
    reg_lambda=1,            
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    use_label_encoder=False
)

In [ ]:
model.fit(X_train,y_train)

In [ ]:
y_pred=model.predict(X_val)

In [ ]:
from sklearn.metrics import classification_report
print("Classification Report:\n", classification_report(y_val, y_pred))
print("Accuracy Score:", accuracy_score(y_val, y_pred))


In [ ]:
test_df["text"] = test_df["prompt"] + ' ' + test_df["response_a"] + ' ' + test_df["response_b"]

# Transform test data using the same vectorizer
X_test = vectorizer.transform(test_df['text'])

In [ ]:
# Predict class probabilities
probs = model.predict_proba(X_test)

# Check a few example rows
probs[:5]


In [ ]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'winner_model_a': probs[:, 0],
    'winner_model_b': probs[:, 1],
    'winner_tie': probs[:, 2],
})

# Save to CSV
submission.to_csv('submission.csv', index=False)

# Preview
submission.head()